# Access the virtual Icechunk Repo FMRC using Rolodex 
This SYFEM coastal model is pushing a netCDF file to object storage each day, and also running the virtualizarr2 append_to_icechunk to update the repo. This notebook opens the up-to-date forecast model run collection and uses Rolodex to extract BestEstimate time series

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
# load AWS credentials for Pangeo-EOSC storage as environment vars
import os
import icechunk
import xarray as xr
from obstore.store import from_url
from dotenv import load_dotenv
_ = load_dotenv(f'{os.environ['HOME']}/dotenv/rsignell4.env')

In [ ]:
# Define storage
storage_endpoint = 'https://pangeo-eosc-minioapi.vm.fedcloud.eu'
storage_bucket = 'rsignell4-protocoast'
storage_name = 'taranto-icechunk-v1'    # 'taranto-icechunk-v1'

## Icechunk concepts

Think of icechunk like a **library**:

| Object | Role |
|---|---|
| `storage` | The building — the physical S3 bucket + path where icechunk metadata lives |
| `RepositoryConfig` | The rules — defines how the repo behaves, including where to find virtual chunk data |
| `Repository` | The catalog — versioned, transactional index of the dataset (like a git repo for arrays) |
| `Session` | A checkout — a consistent, point-in-time view of a branch (e.g. `"main"`) |
| `session.store` | The open book — the zarr-compatible interface that xarray reads from |

**What "virtual" means:** The icechunk repo doesn't copy the raw NetCDF files — it stores *references* to their chunks. The `VirtualChunkContainer` in the config tells icechunk where to fetch those original chunks from when requested, making the repo lightweight while still providing versioning and transactional writes.

In [ ]:
storage = icechunk.s3_storage(
    bucket=storage_bucket,
    prefix=f"icechunk/{storage_name}",
    from_env=True,
    endpoint_url=storage_endpoint,
    region='not-used',   # N/A for Pangeo-EOSC bucket, but required param
    force_path_style=True)

config = icechunk.RepositoryConfig.default()

config.set_virtual_chunk_container(
    icechunk.VirtualChunkContainer(
        url_prefix=f"s3://{storage_bucket}/",
        store=icechunk.s3_store(region="not-used", anonymous=False, s3_compatible=True, 
                                force_path_style=True, endpoint_url=storage_endpoint),
    ),
)

credentials = icechunk.containers_credentials({f"s3://{storage_bucket}/": icechunk.s3_credentials(anonymous=False)})

read_repo = icechunk.Repository.open(storage, config, authorize_virtual_chunk_access=credentials)

read_session = read_repo.readonly_session("main")

In [ ]:
ds = xr.open_zarr(read_session.store, consolidated=False, zarr_format=3, chunks={})
ds

In [ ]:
import rolodex.forecast
from rolodex.forecast import (
    BestEstimate,
    ConstantForecast,
    ConstantOffset,
    ForecastIndex,
    Model,
    ModelRun,
)

In [ ]:
ds.coords["valid_time"] = rolodex.forecast.create_lazy_valid_time_variable(
    reference_time=ds.time, period=ds.step
)

In [ ]:
newds = ds.drop_indexes(["time", "step"]).set_xindex(
    ["time", "step", "valid_time"], ForecastIndex)

In [ ]:
newds

In [ ]:
ds_best = newds.sel(valid_time=BestEstimate(offset=7))  # start at forecast hour 2 instead of 0 (analysis time)

In [ ]:
import hvplot.xarray

In [ ]:
%%time
da = ds_best['temperature'][:,100,0].compute()

In [ ]:
da.hvplot(x='valid_time', grid=True)